# Assignment 4
## Description
In this assignment you must read in a file of metropolitan regions and associated sports teams from [assets/wikipedia_data.html](assets/wikipedia_data.html) and answer some questions about each metropolitan region. Each of these regions may have one or more teams from the "Big 4": NFL (football, in [assets/nfl.csv](assets/nfl.csv)), MLB (baseball, in [assets/mlb.csv](assets/mlb.csv)), NBA (basketball, in [assets/nba.csv](assets/nba.csv) or NHL (hockey, in [assets/nhl.csv](assets/nhl.csv)). Please keep in mind that all questions are from the perspective of the metropolitan region, and that this file is the "source of authority" for the location of a given sports team. Thus teams which are commonly known by a different area (e.g. "Oakland Raiders") need to be mapped into the metropolitan region given (e.g. San Francisco Bay Area). This will require some human data understanding outside of the data you've been given (e.g. you will have to hand-code some names, and might need to google to find out where teams are)!

For each sport I would like you to answer the question: **what is the win/loss ratio's correlation with the population of the city it is in?** Win/Loss ratio refers to the number of wins over the number of wins plus the number of losses. Remember that to calculate the correlation with [`pearsonr`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.pearsonr.html), so you are going to send in two ordered lists of values, the populations from the wikipedia_data.html file and the win/loss ratio for a given sport in the same order. Average the win/loss ratios for those cities which have multiple teams of a single sport. Each sport is worth an equal amount in this assignment (20%\*4=80%) of the grade for this assignment. You should only use data **from year 2018** for your analysis -- this is important!

## Notes

1. Do not include data about the MLS or CFL in any of the work you are doing, we're only interested in the Big 4 in this assignment.
2. I highly suggest that you first tackle the four correlation questions in order, as they are all similar and worth the majority of grades for this assignment. This is by design!
3. It's fair game to talk with peers about high level strategy as well as the relationship between metropolitan areas and sports teams. However, do not post code solving aspects of the assignment (including such as dictionaries mapping areas to teams, or regexes which will clean up names).
4. There may be more teams than the assert statements test, remember to collapse multiple teams in one city into a single value!

As this assignment utilizes global variables in the skeleton code, to avoid having errors in your code you can either:

1. You can place all of your code within the function definitions for all of the questions (other than import statements).
2. You can create copies of all the global variables with the copy() method and proceed as usual.

## Question 1
For this question, calculate the win/loss ratio's correlation with the population of the city it is in for the **NHL** using **2018** data.

In [7]:
# Question 1 Solution
import pandas as pd
import numpy as np
import scipy.stats as stats
import re

def nhl_correlation(): 
    # 1. Load Data
    nhl_df = pd.read_csv("assets/nhl.csv")
    cities = pd.read_html("assets/wikipedia_data.html")[1]
    
    # 2. Clean Cities Data
    # Correct column selection: Metro(0), Pop(3), NFL(5), MLB(6), NBA(7), NHL(8)
    cities = cities.iloc[:-1, [0, 3, 5, 6, 7, 8]]
    cities.columns = ['Metropolitan area', 'Population', 'NFL', 'MLB', 'NBA', 'NHL']
    
    # Clean strings
    cities["Metropolitan area"] = cities["Metropolitan area"].astype(str).str.replace(r'\[.*\]', '', regex=True).str.strip()
    cities["Population"] = cities["Population"].astype(str).str.replace(r'\[.*\]', '', regex=True).str.replace(',', '')
    cities["Population"] = pd.to_numeric(cities["Population"], errors='coerce')
    
    # 3. Clean NHL Data
    nhl_df = nhl_df[nhl_df['year'] == 2018].copy()
    nhl_df['team'] = nhl_df['team'].astype(str).str.replace(r'\*', '', regex=True).str.strip()
    # Filter out division header rows (where 'W' is not a number)
    nhl_df = nhl_df[pd.to_numeric(nhl_df['W'], errors='coerce').notnull()]
    
    # 4. Map Teams to Metro
    team_to_metro = {
        'Tampa Bay Lightning': 'Tampa Bay Area', 'Boston Bruins': 'Boston', 'Toronto Maple Leafs': 'Toronto',
        'Florida Panthers': 'Miami–Fort Lauderdale', 'Detroit Red Wings': 'Detroit', 'Montreal Canadiens': 'Montreal',
        'Ottawa Senators': 'Ottawa', 'Buffalo Sabres': 'Buffalo', 'Washington Capitals': 'Washington, D.C.',
        'Pittsburgh Penguins': 'Pittsburgh', 'Philadelphia Flyers': 'Philadelphia', 'Columbus Blue Jackets': 'Columbus',
        'New Jersey Devils': 'New York City', 'New York Islanders': 'New York City', 'New York Rangers': 'New York City',
        'Carolina Hurricanes': 'Raleigh', 'Nashville Predators': 'Nashville', 'Winnipeg Jets': 'Winnipeg',
        'Minnesota Wild': 'Minneapolis–Saint Paul', 'Colorado Avalanche': 'Denver', 'St. Louis Blues': 'St. Louis',
        'Dallas Stars': 'Dallas–Fort Worth', 'Chicago Blackhawks': 'Chicago', 'Vegas Golden Knights': 'Las Vegas',
        'Anaheim Ducks': 'Los Angeles', 'Los Angeles Kings': 'Los Angeles', 'San Jose Sharks': 'San Francisco Bay Area',
        'Calgary Flames': 'Calgary', 'Edmonton Oilers': 'Edmonton', 'Vancouver Canucks': 'Vancouver', 'Arizona Coyotes': 'Phoenix'
    }
    
    nhl_df['Metropolitan area'] = nhl_df['team'].map(team_to_metro)
    
    # 5. Calculate Ratio
    nhl_df['W'] = pd.to_numeric(nhl_df['W'])
    nhl_df['L'] = pd.to_numeric(nhl_df['L'])
    nhl_df['W_L_Ratio'] = nhl_df['W'] / (nhl_df['W'] + nhl_df['L'])
    
    # 6. Aggregate
    # Drop teams that didn't map to a metro (if any)
    nhl_df = nhl_df.dropna(subset=['Metropolitan area'])
    metro_stats = nhl_df.groupby('Metropolitan area')['W_L_Ratio'].mean().reset_index()
    
    # 7. Merge and Clean Final Data
    merged_df = pd.merge(metro_stats, cities, on='Metropolitan area', how='inner')
    
    # CRITICAL: Drop any rows where Population or Ratio is NaN
    merged_df = merged_df.dropna(subset=['Population', 'W_L_Ratio'])
    
    population_by_region = merged_df['Population']
    win_loss_by_region = merged_df['W_L_Ratio']

    assert len(population_by_region) == len(win_loss_by_region), "Q1: Lists must be same length"
    
    return stats.pearsonr(population_by_region, win_loss_by_region)[0]

## Question 2
For this question, calculate the win/loss ratio's correlation with the population of the city it is in for the **NBA** using **2018** data.

In [8]:
# Question 2 Solution
import pandas as pd
import numpy as np
import scipy.stats as stats
import re

def nba_correlation():
    # 1. Load Data
    nba_df = pd.read_csv("assets/nba.csv")
    cities = pd.read_html("assets/wikipedia_data.html")[1]
    
    # 2. Clean Cities Data
    cities = cities.iloc[:-1, [0, 3, 5, 6, 7, 8]]
    cities.columns = ['Metropolitan area', 'Population', 'NFL', 'MLB', 'NBA', 'NHL']
    
    cities["Metropolitan area"] = cities["Metropolitan area"].astype(str).str.replace(r'\[.*\]', '', regex=True).str.strip()
    cities["Population"] = cities["Population"].astype(str).str.replace(r'\[.*\]', '', regex=True).str.replace(',', '')
    cities["Population"] = pd.to_numeric(cities["Population"], errors='coerce')
    
    # 3. Clean NBA Data
    nba_df = nba_df[nba_df['year'] == 2018].copy()
    # Clean team names: remove * and (text)
    nba_df['team'] = nba_df['team'].str.replace(r'\(.*\)', '', regex=True).str.replace(r'\*', '', regex=True).str.strip()
    nba_df = nba_df[pd.to_numeric(nba_df['W'], errors='coerce').notnull()]
    
    # 4. Map Teams
    team_to_metro = {
        'Toronto Raptors': 'Toronto', 'Boston Celtics': 'Boston', 'Philadelphia 76ers': 'Philadelphia',
        'Cleveland Cavaliers': 'Cleveland', 'Indiana Pacers': 'Indianapolis', 'Miami Heat': 'Miami–Fort Lauderdale',
        'Milwaukee Bucks': 'Milwaukee', 'Washington Wizards': 'Washington, D.C.', 'Detroit Pistons': 'Detroit',
        'Charlotte Hornets': 'Charlotte', 'New York Knicks': 'New York City', 'Brooklyn Nets': 'New York City',
        'Chicago Bulls': 'Chicago', 'Orlando Magic': 'Orlando', 'Atlanta Hawks': 'Atlanta', 'Houston Rockets': 'Houston',
        'Golden State Warriors': 'San Francisco Bay Area', 'Portland Trail Blazers': 'Portland', 'Oklahoma City Thunder': 'Oklahoma City',
        'Utah Jazz': 'Salt Lake City', 'New Orleans Pelicans': 'New Orleans', 'San Antonio Spurs': 'San Antonio',
        'Minnesota Timberwolves': 'Minneapolis–Saint Paul', 'Denver Nuggets': 'Denver', 'Los Angeles Clippers': 'Los Angeles',
        'Los Angeles Lakers': 'Los Angeles', 'Sacramento Kings': 'Sacramento', 'Phoenix Suns': 'Phoenix',
        'Dallas Mavericks': 'Dallas–Fort Worth', 'Memphis Grizzlies': 'Memphis'
    }
    
    nba_df['Metropolitan area'] = nba_df['team'].map(team_to_metro)
    
    # 5. Calculate Ratio
    nba_df['W'] = pd.to_numeric(nba_df['W'])
    nba_df['L'] = pd.to_numeric(nba_df['L'])
    nba_df['W_L_Ratio'] = nba_df['W'] / (nba_df['W'] + nba_df['L'])
    
    # 6. Merge
    nba_df = nba_df.dropna(subset=['Metropolitan area'])
    metro_stats = nba_df.groupby('Metropolitan area')['W_L_Ratio'].mean().reset_index()
    merged_df = pd.merge(metro_stats, cities, on='Metropolitan area', how='inner')
    
    merged_df = merged_df.dropna(subset=['Population', 'W_L_Ratio'])
    
    population_by_region = merged_df['Population']
    win_loss_by_region = merged_df['W_L_Ratio']

    assert len(population_by_region) == len(win_loss_by_region), "Q2: Lists must be same length"

    return stats.pearsonr(population_by_region, win_loss_by_region)[0]

## Question 3
For this question, calculate the win/loss ratio's correlation with the population of the city it is in for the **MLB** using **2018** data.

In [9]:
# Question 3 Solution
import pandas as pd
import numpy as np
import scipy.stats as stats
import re

def mlb_correlation(): 
    # 1. Load Data
    mlb_df = pd.read_csv("assets/mlb.csv")
    cities = pd.read_html("assets/wikipedia_data.html")[1]
    
    # 2. Clean Cities Data
    cities = cities.iloc[:-1, [0, 3, 5, 6, 7, 8]]
    cities.columns = ['Metropolitan area', 'Population', 'NFL', 'MLB', 'NBA', 'NHL']
    
    cities["Metropolitan area"] = cities["Metropolitan area"].astype(str).str.replace(r'\[.*\]', '', regex=True).str.strip()
    cities["Population"] = cities["Population"].astype(str).str.replace(r'\[.*\]', '', regex=True).str.replace(',', '')
    cities["Population"] = pd.to_numeric(cities["Population"], errors='coerce')
    
    # 3. Clean MLB Data
    mlb_df = mlb_df[mlb_df['year'] == 2018].copy()
    mlb_df['team'] = mlb_df['team'].str.strip()
    mlb_df = mlb_df[pd.to_numeric(mlb_df['W'], errors='coerce').notnull()]
    
    # 4. Map Teams
    team_to_metro = {
        'Boston Red Sox': 'Boston', 'New York Yankees': 'New York City', 'Tampa Bay Rays': 'Tampa Bay Area',
        'Toronto Blue Jays': 'Toronto', 'Baltimore Orioles': 'Baltimore', 'Cleveland Indians': 'Cleveland',
        'Minnesota Twins': 'Minneapolis–Saint Paul', 'Detroit Tigers': 'Detroit', 'Chicago White Sox': 'Chicago',
        'Kansas City Royals': 'Kansas City', 'Houston Astros': 'Houston', 'Oakland Athletics': 'San Francisco Bay Area',
        'Seattle Mariners': 'Seattle', 'Los Angeles Angels': 'Los Angeles', 'Texas Rangers': 'Dallas–Fort Worth',
        'Atlanta Braves': 'Atlanta', 'Washington Nationals': 'Washington, D.C.', 'Philadelphia Phillies': 'Philadelphia',
        'New York Mets': 'New York City', 'Miami Marlins': 'Miami–Fort Lauderdale', 'Milwaukee Brewers': 'Milwaukee',
        'St. Louis Cardinals': 'St. Louis', 'Chicago Cubs': 'Chicago', 'Arizona Diamondbacks': 'Phoenix',
        'Los Angeles Dodgers': 'Los Angeles', 'San Francisco Giants': 'San Francisco Bay Area', 'San Diego Padres': 'San Diego',
        'Colorado Rockies': 'Denver', 'Cincinnati Reds': 'Cincinnati', 'Pittsburgh Pirates': 'Pittsburgh'
    }
    
    mlb_df['Metropolitan area'] = mlb_df['team'].map(team_to_metro)
    
    # 5. Calculate Ratio
    mlb_df['W'] = pd.to_numeric(mlb_df['W'])
    mlb_df['L'] = pd.to_numeric(mlb_df['L'])
    mlb_df['W_L_Ratio'] = mlb_df['W'] / (mlb_df['W'] + mlb_df['L'])
    
    # 6. Merge
    mlb_df = mlb_df.dropna(subset=['Metropolitan area'])
    metro_stats = mlb_df.groupby('Metropolitan area')['W_L_Ratio'].mean().reset_index()
    merged_df = pd.merge(metro_stats, cities, on='Metropolitan area', how='inner')
    
    merged_df = merged_df.dropna(subset=['Population', 'W_L_Ratio'])
    
    population_by_region = merged_df['Population']
    win_loss_by_region = merged_df['W_L_Ratio']
    
    assert len(population_by_region) == len(win_loss_by_region), "Q3: Lists must be same length"

    return stats.pearsonr(population_by_region, win_loss_by_region)[0]

## Question 4
For this question, calculate the win/loss ratio's correlation with the population of the city it is in for the **NFL** using **2018** data.

In [10]:
# Question 4 Solution
import pandas as pd
import numpy as np
import scipy.stats as stats
import re

def nfl_correlation(): 
    # 1. Load Data
    nfl_df = pd.read_csv("assets/nfl.csv")
    cities = pd.read_html("assets/wikipedia_data.html")[1]
    
    # 2. Clean Cities Data
    cities = cities.iloc[:-1, [0, 3, 5, 6, 7, 8]]
    cities.columns = ['Metropolitan area', 'Population', 'NFL', 'MLB', 'NBA', 'NHL']
    
    cities["Metropolitan area"] = cities["Metropolitan area"].astype(str).str.replace(r'\[.*\]', '', regex=True).str.strip()
    cities["Population"] = cities["Population"].astype(str).str.replace(r'\[.*\]', '', regex=True).str.replace(',', '')
    cities["Population"] = pd.to_numeric(cities["Population"], errors='coerce')
    
    # 3. Clean NFL Data
    nfl_df = nfl_df[nfl_df['year'] == 2018].copy()
    nfl_df['team'] = nfl_df['team'].str.replace(r'[\*\+]', '', regex=True).str.strip()
    nfl_df = nfl_df[pd.to_numeric(nfl_df['W'], errors='coerce').notnull()]
    
    # 4. Map Teams
    team_to_metro = {
        'New England Patriots': 'Boston', 'Buffalo Bills': 'Buffalo', 'New York Jets': 'New York City',
        'New York Giants': 'New York City', 'Miami Dolphins': 'Miami–Fort Lauderdale', 'Tennessee Titans': 'Nashville',
        'Jacksonville Jaguars': 'Jacksonville', 'Houston Texans': 'Houston', 'Indianapolis Colts': 'Indianapolis',
        'Pittsburgh Steelers': 'Pittsburgh', 'Baltimore Ravens': 'Baltimore', 'Cleveland Browns': 'Cleveland',
        'Cincinnati Bengals': 'Cincinnati', 'Kansas City Chiefs': 'Kansas City', 'Los Angeles Chargers': 'Los Angeles',
        'Oakland Raiders': 'San Francisco Bay Area', 'Denver Broncos': 'Denver', 'New Orleans Saints': 'New Orleans',
        'Carolina Panthers': 'Charlotte', 'Atlanta Falcons': 'Atlanta', 'Tampa Bay Buccaneers': 'Tampa Bay Area',
        'Los Angeles Rams': 'Los Angeles', 'Seattle Seahawks': 'Seattle', 'San Francisco 49ers': 'San Francisco Bay Area',
        'Arizona Cardinals': 'Phoenix', 'Chicago Bears': 'Chicago', 'Minnesota Vikings': 'Minneapolis–Saint Paul',
        'Green Bay Packers': 'Green Bay', 'Detroit Lions': 'Detroit', 'Philadelphia Eagles': 'Philadelphia',
        'Dallas Cowboys': 'Dallas–Fort Worth', 'Washington Redskins': 'Washington, D.C.'
    }
    
    nfl_df['Metropolitan area'] = nfl_df['team'].map(team_to_metro)
    
    # 5. Calculate Ratio
    nfl_df['W'] = pd.to_numeric(nfl_df['W'])
    nfl_df['L'] = pd.to_numeric(nfl_df['L'])
    nfl_df['W_L_Ratio'] = nfl_df['W'] / (nfl_df['W'] + nfl_df['L'])
    
    # 6. Merge
    nfl_df = nfl_df.dropna(subset=['Metropolitan area'])
    metro_stats = nfl_df.groupby('Metropolitan area')['W_L_Ratio'].mean().reset_index()
    merged_df = pd.merge(metro_stats, cities, on='Metropolitan area', how='inner')
    
    merged_df = merged_df.dropna(subset=['Population', 'W_L_Ratio'])
    
    population_by_region = merged_df['Population']
    win_loss_by_region = merged_df['W_L_Ratio']
    
    assert len(population_by_region) == len(win_loss_by_region), "Q4: Lists must be same length"

    return stats.pearsonr(population_by_region, win_loss_by_region)[0]

## Question 5
In this question I would like you to explore the hypothesis that **given that an area has two sports teams in different sports, those teams will perform the same within their respective sports**. How I would like to see this explored is with a series of paired t-tests (so use [`ttest_rel`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_rel.html)) between all pairs of sports. Are there any sports where we can reject the null hypothesis? Again, average values where a sport has multiple teams in one region. Remember, you will only be including, for each sport, cities which have teams engaged in that sport, drop others as appropriate. This question is worth 20% of the grade for this assignment.

In [10]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import re

mlb_df=pd.read_csv("assets/mlb.csv")
nhl_df=pd.read_csv("assets/nhl.csv")
nba_df=pd.read_csv("assets/nba.csv")
nfl_df=pd.read_csv("assets/nfl.csv")
cities=pd.read_html("assets/wikipedia_data.html")[1]
cities=cities.iloc[:-1,[0,3,5,6,7,8]]

def sports_team_performance():
    # YOUR CODE HERE
    
    # 1. Define Helper to Process Each Sport
    def get_sport_data(sport_name):
        
        if sport_name == 'NFL':
            df = nfl_df[nfl_df['year'] == 2018].copy()
            df['team'] = df['team'].str.replace(r'[\*\+]', '', regex=True)
            df = df[pd.to_numeric(df['W'], errors='coerce').notnull()] # Remove headers
            mapping = {
                'New England Patriots': 'Boston', 'Buffalo Bills': 'Buffalo', 'New York Jets': 'New York City',
                'New York Giants': 'New York City', 'Miami Dolphins': 'Miami–Fort Lauderdale', 'Tennessee Titans': 'Nashville',
                'Jacksonville Jaguars': 'Jacksonville', 'Houston Texans': 'Houston', 'Indianapolis Colts': 'Indianapolis',
                'Pittsburgh Steelers': 'Pittsburgh', 'Baltimore Ravens': 'Baltimore', 'Cleveland Browns': 'Cleveland',
                'Cincinnati Bengals': 'Cincinnati', 'Kansas City Chiefs': 'Kansas City', 'Los Angeles Chargers': 'Los Angeles',
                'Oakland Raiders': 'San Francisco Bay Area', 'Denver Broncos': 'Denver', 'New Orleans Saints': 'New Orleans',
                'Carolina Panthers': 'Charlotte', 'Atlanta Falcons': 'Atlanta', 'Tampa Bay Buccaneers': 'Tampa Bay Area',
                'Los Angeles Rams': 'Los Angeles', 'Seattle Seahawks': 'Seattle', 'San Francisco 49ers': 'San Francisco Bay Area',
                'Arizona Cardinals': 'Phoenix', 'Chicago Bears': 'Chicago', 'Minnesota Vikings': 'Minneapolis–Saint Paul',
                'Green Bay Packers': 'Green Bay', 'Detroit Lions': 'Detroit', 'Philadelphia Eagles': 'Philadelphia',
                'Dallas Cowboys': 'Dallas–Fort Worth', 'Washington Redskins': 'Washington, D.C.'
            }
            
        elif sport_name == 'NBA':
            df = nba_df[nba_df['year'] == 2018].copy()
            df['team'] = df['team'].str.replace(r'[\*\(].*', '', regex=True).str.strip()
            df = df[pd.to_numeric(df['W'], errors='coerce').notnull()]
            mapping = {
                'Toronto Raptors': 'Toronto', 'Boston Celtics': 'Boston', 'Philadelphia 76ers': 'Philadelphia',
                'Cleveland Cavaliers': 'Cleveland', 'Indiana Pacers': 'Indianapolis', 'Miami Heat': 'Miami–Fort Lauderdale',
                'Milwaukee Bucks': 'Milwaukee', 'Washington Wizards': 'Washington, D.C.', 'Detroit Pistons': 'Detroit',
                'Charlotte Hornets': 'Charlotte', 'New York Knicks': 'New York City', 'Brooklyn Nets': 'New York City',
                'Chicago Bulls': 'Chicago', 'Orlando Magic': 'Orlando', 'Atlanta Hawks': 'Atlanta', 'Houston Rockets': 'Houston',
                'Golden State Warriors': 'San Francisco Bay Area', 'Portland Trail Blazers': 'Portland', 'Oklahoma City Thunder': 'Oklahoma City',
                'Utah Jazz': 'Salt Lake City', 'New Orleans Pelicans': 'New Orleans', 'San Antonio Spurs': 'San Antonio',
                'Minnesota Timberwolves': 'Minneapolis–Saint Paul', 'Denver Nuggets': 'Denver', 'Los Angeles Clippers': 'Los Angeles',
                'Los Angeles Lakers': 'Los Angeles', 'Sacramento Kings': 'Sacramento', 'Phoenix Suns': 'Phoenix',
                'Dallas Mavericks': 'Dallas–Fort Worth', 'Memphis Grizzlies': 'Memphis'
            }
            
        elif sport_name == 'NHL':
            df = nhl_df[nhl_df['year'] == 2018].copy()
            df['team'] = df['team'].str.replace(r'\*', '', regex=True)
            df = df[df['team'].str.contains('Division') == False]
            mapping = {
                'Tampa Bay Lightning': 'Tampa Bay Area', 'Boston Bruins': 'Boston', 'Toronto Maple Leafs': 'Toronto',
                'Florida Panthers': 'Miami–Fort Lauderdale', 'Detroit Red Wings': 'Detroit', 'Montreal Canadiens': 'Montreal',
                'Ottawa Senators': 'Ottawa', 'Buffalo Sabres': 'Buffalo', 'Washington Capitals': 'Washington, D.C.',
                'Pittsburgh Penguins': 'Pittsburgh', 'Philadelphia Flyers': 'Philadelphia', 'Columbus Blue Jackets': 'Columbus',
                'New Jersey Devils': 'New York City', 'New York Islanders': 'New York City', 'New York Rangers': 'New York City',
                'Carolina Hurricanes': 'Raleigh', 'Nashville Predators': 'Nashville', 'Winnipeg Jets': 'Winnipeg',
                'Minnesota Wild': 'Minneapolis–Saint Paul', 'Colorado Avalanche': 'Denver', 'St. Louis Blues': 'St. Louis',
                'Dallas Stars': 'Dallas–Fort Worth', 'Chicago Blackhawks': 'Chicago', 'Vegas Golden Knights': 'Las Vegas',
                'Anaheim Ducks': 'Los Angeles', 'Los Angeles Kings': 'Los Angeles', 'San Jose Sharks': 'San Francisco Bay Area',
                'Calgary Flames': 'Calgary', 'Edmonton Oilers': 'Edmonton', 'Vancouver Canucks': 'Vancouver', 'Arizona Coyotes': 'Phoenix'
            }
            
        elif sport_name == 'MLB':
            df = mlb_df[mlb_df['year'] == 2018].copy()
            df = df[pd.to_numeric(df['W'], errors='coerce').notnull()]
            mapping = {
                'Boston Red Sox': 'Boston', 'New York Yankees': 'New York City', 'Tampa Bay Rays': 'Tampa Bay Area',
                'Toronto Blue Jays': 'Toronto', 'Baltimore Orioles': 'Baltimore', 'Cleveland Indians': 'Cleveland',
                'Minnesota Twins': 'Minneapolis–Saint Paul', 'Detroit Tigers': 'Detroit', 'Chicago White Sox': 'Chicago',
                'Kansas City Royals': 'Kansas City', 'Houston Astros': 'Houston', 'Oakland Athletics': 'San Francisco Bay Area',
                'Seattle Mariners': 'Seattle', 'Los Angeles Angels': 'Los Angeles', 'Texas Rangers': 'Dallas–Fort Worth',
                'Atlanta Braves': 'Atlanta', 'Washington Nationals': 'Washington, D.C.', 'Philadelphia Phillies': 'Philadelphia',
                'New York Mets': 'New York City', 'Miami Marlins': 'Miami–Fort Lauderdale', 'Milwaukee Brewers': 'Milwaukee',
                'St. Louis Cardinals': 'St. Louis', 'Chicago Cubs': 'Chicago', 'Arizona Diamondbacks': 'Phoenix',
                'Los Angeles Dodgers': 'Los Angeles', 'San Francisco Giants': 'San Francisco Bay Area', 'San Diego Padres': 'San Diego',
                'Colorado Rockies': 'Denver', 'Cincinnati Reds': 'Cincinnati', 'Pittsburgh Pirates': 'Pittsburgh'
            }

        # Apply Mapping
        df['Metropolitan area'] = df['team'].map(mapping)
        
        # Calculate Ratio
        df['W'] = pd.to_numeric(df['W'])
        df['L'] = pd.to_numeric(df['L'])
        df['W_L_Ratio'] = df['W'] / (df['W'] + df['L'])
        
        # Aggregate by Metro Area
        return df.groupby('Metropolitan area')['W_L_Ratio'].mean()

    # 2. Get data for all sports
    sports = ['NFL', 'NBA', 'NHL', 'MLB']
    data = {sport: get_sport_data(sport) for sport in sports}
    
    # 3. Create P-Values DataFrame
    p_values = pd.DataFrame({k: np.nan for k in sports}, index=sports)
    
    # 4. Perform Paired T-Tests
    for i in sports:
        for j in sports:
            if i != j:
                # Merge the two series on Index (Metropolitan area)
                # inner join keeps only cities that have teams in BOTH sports
                merged = pd.merge(data[i], data[j], how='inner', left_index=True, right_index=True)
                
                # Run t-test
                p_val = stats.ttest_rel(merged['W_L_Ratio_x'], merged['W_L_Ratio_y'])[1]
                p_values.loc[i, j] = p_val
                
    
    assert abs(p_values.loc["NBA", "NHL"] - 0.02) <= 1e-2, "The NBA-NHL p-value should be around 0.02"
    assert abs(p_values.loc["MLB", "NFL"] - 0.80) <= 1e-2, "The MLB-NFL p-value should be around 0.80"
    
    return p_values    
    # Note: p_values is a full dataframe, so df.loc["NFL","NBA"] should be the same as df.loc["NBA","NFL"] and
    # df.loc["NFL","NFL"] should return np.nan
    sports = ['NFL', 'NBA', 'NHL', 'MLB']
    p_values = pd.DataFrame({k:np.nan for k in sports}, index=sports)
    
    assert abs(p_values.loc["NBA", "NHL"] - 0.02) <= 1e-2, "The NBA-NHL p-value should be around 0.02"
    assert abs(p_values.loc["MLB", "NFL"] - 0.80) <= 1e-2, "The MLB-NFL p-value should be around 0.80"
    return p_values